Baseline-модель для предсказания медианной стоимости жилья в районах Калифорнии 
на основе 8 признаков: медианного дохода жителей, возраста застройки, среднего 
числа комнат, плотности населения и географических координат.

**Практическая ценность:** оценка стоимости недвижимости — типовая задача в 
банках (ипотечный скоринг), агентствах недвижимости, страховых компаниях и 
муниципальном планировании. Линейная регрессия даёт интерпретируемую модель: 
видно, как именно каждый фактор влияет на цену, что важно для обоснования 
решений перед бизнесом или клиентом.

In [ ]:
!pip3 install pandas scikit-learn matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

## 1. Загрузка данных

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame
df.head()

## 2. Общая информация

In [ ]:
print(df.shape)
df.describe()

In [ ]:
df.isnull().sum()

## 3. Распределение целевой переменной

In [ ]:
df['MedHouseVal'].hist(bins=50)
plt.xlabel('Median House Value')
plt.title('Распределение целевой переменной')
plt.show()

**Наблюдения:** распределение скошено вправо, мода около $150k. 
Заметна искусственная отсечка на 5.0 (~1000 районов) — известный 
артефакт датасета: оригинальные цены выше $500k обрезаны.

## 4. Корреляции с таргетом

In [ ]:
df.corr()['MedHouseVal'].sort_values(ascending=False)

## 5. Доход vs цена

In [ ]:
plt.scatter(df['MedInc'], df['MedHouseVal'], alpha=0.2, s=3)
plt.xlabel('Медианный доход')
plt.ylabel('Цена дома')
plt.title('Доход vs Цена')
plt.show()

## 6. Train/test split

In [ ]:
X = df.drop(['MedHouseVal'], axis=1)
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 7. Обучение модели и метрики

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae:.3f}')
print(f'R2: {r2:.3f}')

## 8. Анализ коэффициентов

In [ ]:
coefs = pd.DataFrame({
    'feature': X.columns,
    'coef': model.coef_
}).sort_values('coef', key=abs, ascending=False)
print(coefs)

## 9. Предсказания vs реальные значения

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.2, s=5)
plt.plot([0, 5], [0, 5], 'r--')
plt.xlabel('Реальная цена')
plt.ylabel('Предсказанная цена')
plt.title('Предсказания vs реальные значения')
plt.show()